In [ ]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import os
import matplotlib.pyplot as plt
from plyfile import PlyData, PlyElement
import numpy as np
import mitsuba as mi
import random
import json

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies
 
#NOTE: BEFORE RUNNING THE PROGRAM, PLEASE MAKE SURE THAT THE XML FILE CAN BE FOUND IN /sionna/rt/scene.py

# Path to the sionna installation, change accordingly
SIONNAPATH = '/home/erasmus/miniconda3/envs/sionna_1_2_0/lib/python3.11/site-packages/sionna/rt/'

# General environment settings
ITERATIONS = 1
MINLENGTH = 35
MAXLENGTH = 40
MINWIDTH = 35   
MAXWIDTH = 40
MINHEIGHT = 35
MAXHEIGHT = 40
MATERIALS = ['glass', 'wood', 'marble', 'brick', 'concrete', 'metal'] # Other possible materials to add: ['ceiling_board', 'chipboard', 'plasterboard', 'plywood', 'very_dry_ground', 'medium_dry_ground', 'wet_ground']
MAXDEPTH = 3 # How many max refrations/diffractions

# Use the same material for all surfaces
SAMEMATERIAL = True

jsonarray = []

for i in range(0, ITERATIONS):

    # Random size for the box
    boxlength = random.uniform(MINLENGTH, MAXLENGTH)
    boxwidth = random.uniform(MINWIDTH, MAXWIDTH)
    boxheight = random.uniform(MINHEIGHT, MAXHEIGHT)

    if SAMEMATERIAL:
        floormaterial = random.choice(MATERIALS)
        wallmaterial = floormaterial
        roofmaterial = floormaterial
    else:
        floormaterial = random.choice(MATERIALS)
        wallmaterial = random.choice(MATERIALS)
        roofmaterial = random.choice(MATERIALS)

    # Creates the ply and xml files needed for the scene
    if not os.path.isdir(SIONNAPATH + 'scenes/project/'):
        os.makedirs(SIONNAPATH + 'scenes/project/meshes', exist_ok=True)

        # Instantiate a 1x1 size cube for easier translation
        floorvertices = np.array([(0.5, 0.5, 0.0),
                                  (-0.5, -0.5, 0.0),
                                  (-0.5, 0.5, 0.0),
                                  (0.5, -0.5, 0.0)],
                                dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])

        floorfaces = np.array([([0, 1, 2], 0, 0, 0),
                               ([0, 3, 1], 0, 0, 0)],
                            dtype=[('vertex_indices', 'i4', (3,)),
                                    ('red', 'f4'), ('green', 'f4'),
                                    ('blue', 'f4')])

        floorverticeselement = PlyElement.describe(floorvertices, 'vertex')
        floorfaceselement = PlyElement.describe(floorfaces, 'face',
                                                val_types={'vertex_indices': 'u2'},
                                                len_types={'vertex_indices': 'u4'})
        
        wall1vertices = np.array([(-0.5, 0.5, 0.5),
                                  (0.5, 0.5, 0.0),
                                  (-0.5, 0.5, 0.0),
                                  (0.5, 0.5, 0.5)],
                                dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])

        wall1faces = np.array([([0, 1, 2], 0, 0, 0),
                               ([0, 3, 1], 0, 0, 0)],
                            dtype=[('vertex_indices', 'i4', (3,)),
                                   ('red', 'f4'), ('green', 'f4'),
                                   ('blue', 'f4')])

        wall1verticeselement = PlyElement.describe(wall1vertices, 'vertex')
        wall1faceselement = PlyElement.describe(wall1faces, 'face',
                                                val_types={'vertex_indices': 'u2'},
                                                len_types={'vertex_indices': 'u4'})
        
        wall2vertices = np.array([(0.5, 0.5, 0.5),
                                  (0.5, -0.5, 0.0),
                                  (0.5, 0.5, 0.0),
                                  (0.5, -0.5, 0.5)],
                                dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])

        wall2faces = np.array([([0, 1, 2], 0, 0, 0),
                               ([0, 3, 1], 0, 0, 0)],
                            dtype=[('vertex_indices', 'i4', (3,)),
                                   ('red', 'f4'), ('green', 'f4'),
                                   ('blue', 'f4')])

        wall2verticeselement = PlyElement.describe(wall2vertices, 'vertex')
        wall2faceselement = PlyElement.describe(wall2faces, 'face',
                                                val_types={'vertex_indices': 'u2'},
                                                len_types={'vertex_indices': 'u4'})


        wall3vertices = np.array([(0.5, -0.5, 0.5),
                                  (-0.5, -0.5, 0.0),
                                  (0.5, -0.5, 0.0),
                                  (-0.5, -0.5, 0.5)],
                                dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])

        wall3faces = np.array([([0, 1, 2], 0, 0, 0),
                               ([0, 3, 1], 0, 0, 0)],
                            dtype=[('vertex_indices', 'i4', (3,)),
                                   ('red', 'f4'), ('green', 'f4'),
                                   ('blue', 'f4')])

        wall3verticeselement = PlyElement.describe(wall3vertices, 'vertex')
        wall3faceselement = PlyElement.describe(wall3faces, 'face',
                                                val_types={'vertex_indices': 'u2'},
                                                len_types={'vertex_indices': 'u4'})


        wall4vertices = np.array([(-0.5, -0.5, 0.5),
                                  (-0.5, 0.5, 0.0),
                                  (-0.5, -0.5, 0.0),
                                  (-0.5, 0.5, 0.5)],
                                dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])

        wall4faces = np.array([([0, 1, 2], 0, 0, 0),
                               ([0, 3, 1], 0, 0, 0)],
                            dtype=[('vertex_indices', 'i4', (3,)),
                                   ('red', 'f4'), ('green', 'f4'),
                                   ('blue', 'f4')])

        wall4verticeselement = PlyElement.describe(wall4vertices, 'vertex')
        wall4faceselement = PlyElement.describe(wall4faces, 'face',
                                                val_types={'vertex_indices': 'u2'},
                                                len_types={'vertex_indices': 'u4'})


        roofvertices = np.array([(-0.5, -0.5, 0.5),
                                  (0.5, 0.5, 0.5),
                                  (-0.5, 0.5, 0.5),
                                  (0.5, -0.5, 0.5)],
                                dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4')])

        rooffaces = np.array([([0, 1, 2], 0, 0, 0),
                              ([0, 3, 1], 0, 0, 0)],
                            dtype=[('vertex_indices', 'i4', (3,)),
                                  ('red', 'f4'), ('green', 'f4'),
                                  ('blue', 'f4')])

        roofverticeselement = PlyElement.describe(roofvertices, 'vertex')
        rooffaceselement = PlyElement.describe(rooffaces, 'face',
                                               val_types={'vertex_indices': 'u2'},
                                               len_types={'vertex_indices': 'u4'})
        
        PlyData([floorverticeselement, floorfaceselement]).write(SIONNAPATH + 'scenes/project/meshes/floor.ply')
        PlyData([wall1verticeselement, wall1faceselement]).write(SIONNAPATH + 'scenes/project/meshes/wall1.ply')
        PlyData([wall2verticeselement, wall2faceselement]).write(SIONNAPATH + 'scenes/project/meshes/wall2.ply')
        PlyData([wall3verticeselement, wall3faceselement]).write(SIONNAPATH + 'scenes/project/meshes/wall3.ply')
        PlyData([wall4verticeselement, wall4faceselement]).write(SIONNAPATH + 'scenes/project/meshes/wall4.ply')
        PlyData([roofverticeselement, rooffaceselement]).write(SIONNAPATH + 'scenes/project/meshes/roof.ply')


    with open(SIONNAPATH + 'scenes/project/project.xml', "w") as f:
        f.write(''' 
<scene version="2.1.0">

    <!-- Materials -->

    <bsdf type="itu-radio-material" id="glass">
        <string name="type" value="glass"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="wood">
        <string name="type" value="wood"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="marble">
        <string name="type" value="marble"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="brick">
        <string name="type" value="brick"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="concrete">
        <string name="type" value="concrete"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="metal">
        <string name="type" value="metal"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="ceiling_board">
        <string name="type" value="ceiling_board"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="chipboard">
        <string name="type" value="chipboard"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="plasterboard">
        <string name="type" value="plasterboard"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="plywood">
        <string name="type" value="plywood"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="very_dry_ground">
        <string name="type" value="very_dry_ground"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="medium_dry_ground">
        <string name="type" value="medium_dry_ground"/>
        <float name="thickness" value="0.1"/>
    </bsdf>
    <bsdf type="itu-radio-material" id="wet_ground">
        <string name="type" value="wet_ground"/>
        <float name="thickness" value="0.1"/>
    </bsdf>

    <!-- Shapes -->

    <shape type="ply" id="floor">
        <string name="filename" value="meshes/floor.ply"/>
        <boolean name="face_normals" value="true"/>
        <ref id="''' + floormaterial + '''" name="bsdf"/>
    </shape>
    <shape type="ply" id="wall1">
        <string name="filename" value="meshes/wall1.ply"/>
        <boolean name="face_normals" value="true"/>
        <ref id="''' + wallmaterial + '''" name="bsdf"/>
    </shape>
    <shape type="ply" id="wall2">
        <string name="filename" value="meshes/wall2.ply"/>
        <boolean name="face_normals" value="true"/>
        <ref id="''' + wallmaterial + '''" name="bsdf"/>
    </shape>
    <shape type="ply" id="wall3">
        <string name="filename" value="meshes/wall3.ply"/>
        <boolean name="face_normals" value="true"/>
        <ref id="''' + wallmaterial + '''" name="bsdf"/>
    </shape>
    <shape type="ply" id="wall4">
        <string name="filename" value="meshes/wall4.ply"/>
        <boolean name="face_normals" value="true"/>
        <ref id="''' + wallmaterial + '''" name="bsdf"/>
    </shape>
    <shape type="ply" id="roof">
        <string name="filename" value="meshes/roof.ply"/>
        <boolean name="face_normals" value="true"/>
        <ref id="''' + roofmaterial + '''" name="bsdf"/>
    </shape>
</scene>''')
            

    # Load the scene
    scene = load_scene(sionna.rt.scene.project, merge_shapes=False)

    # Change the box's dimensions to the random sizes generated above
    floor = scene.get("floor")
    floor.scaling = [boxlength, boxwidth, 1]
    floorscaling = floor.scaling

    roof = scene.get("roof")
    roofposition = roof.position
    roof.scaling = floor.scaling
    roof.position = [roofposition[0], roofposition[1], boxheight]

    wall1 = scene.get("wall1")
    wall1position = wall1.position
    wall1scaling = wall1.scaling
    wall1.scaling = [floorscaling[0], wall1scaling[1], 2*boxheight]
    wall1.position = [wall1position[0], floorscaling[1] / 2, wall1.scaling[2] / 4]

    wall2 = scene.get("wall2")
    wall2position = wall2.position
    wall2scaling = wall2.scaling
    wall2.scaling = [wall2scaling[0], floorscaling[1], 2*boxheight]
    wall2.position = [floorscaling[0] / 2, wall2position[1], wall2.scaling[2] / 4]

    wall3 = scene.get("wall3")
    wall3scaling = wall3.scaling
    wall3position = wall3.position
    wall3.scaling = wall1.scaling
    wall3.position = [wall1.position[0], -wall1.position[1], wall1.position[2]]

    wall4 = scene.get("wall4")
    wall4scaling = wall4.scaling
    wall4position = wall4.position
    wall4.scaling = wall2.scaling
    wall4.position = [-wall2.position[0], wall2.position[1], wall2.position[2]]


    # Creating the transmitter

    transmitterx = random.uniform(-boxlength/2, boxlength/2)
    transmittery = random.uniform(-boxwidth/2, boxwidth/2)
    transmitterz = random.uniform(0, boxheight)
    # print("transmitter: (x, y, z)", transmitterx, transmittery, transmitterz)

    tx = Transmitter(name="tx", position=[transmitterx, transmittery, transmitterz], display_radius=2)
    scene.add(tx)

    scene.tx_array = PlanarArray(num_rows=1,
                                num_cols=1,
                                vertical_spacing=0.5,
                                horizontal_spacing=0.5,
                                pattern="tr38901",
                                polarization="V")
    
    # Creating the receiver

    receiverx = random.uniform(-boxlength/2, boxlength/2)
    receivery = random.uniform(-boxwidth/2, boxwidth/2)
    receiverz = random.uniform(0, boxheight)
    # print("receiver: (x, y, z)", receiverx, receivery, receiverz)

    rx = Receiver(name="rx", position=[receiverx, receivery, receiverz], display_radius=2)
    scene.add(rx)

    tx.look_at(rx)

    scene.rx_array = PlanarArray(num_rows=1,
                                num_cols=1,
                                vertical_spacing=0.5,
                                horizontal_spacing=0.5,
                                pattern="dipole",
                                polarization="cross")

    # Compute Path solver
    p_solver = PathSolver()
    paths = p_solver(scene=scene,
                    max_depth=MAXDEPTH,
                    los=True,
                    specular_reflection=True,
                    diffuse_reflection=False,
                    refraction=True,
                    synthetic_array=False,
                    seed=41)

    # Compute values for the channel impulse response
    a, tau = paths.cir(normalize_delays=True, out_type="numpy")

    t = tau[0,0,0,0,:]/1e-9
    a_abs = np.abs(a)[0,0,0,0,:,0]
    a_max = np.max(a_abs)

    atau = []
    meanpower = 0
    meanpowerdelay = 0
    meanpowerdelaysquare = 0

    for tauvalue, avalue in zip(t, a_abs):
        avalue = sionna.phy.utils.dbm_to_watt(avalue, precision=None)       
        meanpower += avalue
        meanpowerdelay += avalue * tauvalue
        meanpowerdelaysquare += avalue * (tauvalue * tauvalue)

        formattedtauvalue = "{:.10f}".format(tauvalue)
        formattedavalue = "{:.10f}".format(avalue)
        atau.append((formattedtauvalue, formattedavalue))

    meandelay = meanpowerdelay / meanpower
    rmsdelayspread = np.sqrt((meanpowerdelaysquare / meanpower) - (meandelay * meandelay))

    # Find the path angles

    phi_r = paths.phi_r.numpy()[0,0,0,0,:]

    # Compute the RMS angular spread

    # We computed the arrival angle by averaging the value for cos(φ) + j sin(φ)

    powers = []

    for avalue in a_abs:
        p = sionna.phy.utils.dbm_to_watt(avalue, precision=None)
        powers.append(p)

    powers = np.array(powers)

    mean_cos = np.sum(powers * np.cos(phi_r)) / np.sum(powers)
    mean_sin = np.sum(powers * np.sin(phi_r)) / np.sum(powers)

    mean_phi = np.arctan2(mean_sin, mean_cos)

    def angle_diff(a, b):
        return np.angle(np.exp(1j*(a-b)))
    diff = angle_diff(phi_r, mean_phi)

    rmsangularspread = np.sqrt(
        np.sum(powers * diff**2) / np.sum(powers)
    )


    # Conversion from radians to degrees if needed:
    rmsangularspread = np.degrees(rmsangularspread)



    # Compute values for the channel frequency response
    num_subcarriers = 1024
    subcarrier_spacing=30e3
    frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)

    h_freq = paths.cfr(frequencies=frequencies,
                       normalize=True, # Normalize energy
                       normalize_delays=True,
                       out_type="numpy")

    # Add iteration data to a dictionary for the json output    
    data = {
        "id": i,
        "length": "{:.3f}".format(boxlength),
        "width": "{:.3f}".format(boxwidth),
        "height": "{:.3f}".format(boxheight),
        "floor_material": floormaterial,
        "wall_material": wallmaterial,
        "roof_material": roofmaterial,
        "transmitter_pos": {
            "x": "{:.3f}".format(transmitterx),
            "y": "{:.3f}".format(transmittery),
            "z": "{:.3f}".format(transmitterz)
        },
        "receiver_pos": {
            "x": "{:.3f}".format(receiverx),
            "y": "{:.3f}".format(receivery),
            "z": "{:.3f}".format(receiverz)
        },
        "(tau, a)": atau,
        "meanpower": "{:.6f}".format(meanpower),
        "rmsdelayspread": "{:.3f}".format(rmsdelayspread),
        "rmsangularspread": "{:.3f}".format(rmsangularspread)
    }

    jsonarray.append(data)

    # Scene previews: (uncomment if you want, but the time to compute will be slightly longer)
    # print("Iteration: ", i)
    # scene.preview(paths=paths, clip_at=MAXHEIGHT+2)

# Add all the data to the output
with open("output.json", "w") as f:
    json.dump(jsonarray, f, indent=4)


# If you just want the last iteration's scene preview (remove the preview from the loop):
scene.preview(paths=paths, clip_at=MAXHEIGHT+2)